# Import Libraries

In [1]:
import pandas as pd
import torch
import pickle
import numpy as np
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\VICTUS\OneDrive\Documents\Semester_4\ML\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Models

In [2]:
path_model = '../models/bert' 
tokenizer = DistilBertTokenizer.from_pretrained(path_model)
model = DistilBertForSequenceClassification.from_pretrained(path_model)

path_encoder = '../models/bert/label_encoder.pkl'
with open(path_encoder, 'rb') as f:
    label_encoder = pickle.load(f)

# Load Dataset

In [3]:
df_movies = pd.read_csv('../data/imdb_movies_with_emotions.csv')
df_movies.head(5)

,Series_Title,Genre,Overview,clean_overview,predicted_emotion
0,The Shawshank Redemption,Drama,Two imprisoned men bond over a number of years...,two imprisoned men bond over a number of years...,Sadness
1,The Godfather,"Crime, Drama",An organized crime dynasty's aging patriarch t...,an organized crime dynasty's aging patriarch t...,Anger
2,The Dark Knight,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...,when the menace known as the joker wreaks havo...,Fear
3,The Godfather: Part II,"Crime, Drama",The early life and career of Vito Corleone in ...,the early life and career of vito corleone in ...,Joy
4,12 Angry Men,"Crime, Drama",A jury holdout attempts to prevent a miscarria...,a jury holdout attempts to prevent a miscarria...,Sadness


# TF-IDF Vectorizer

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_vectorizer.fit(df_movies['clean_overview'])

TfidfVectorizer(stop_words='english')

# Emotion Detection

In [5]:
def get_user_emotion(text):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction_idx = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([prediction_idx])[0]

# Recommendation System

In [6]:
def get_movie_recommendations(user_input, top_n=5):
    print(f"\nUser Input: '{user_input}'")
    
    # Step 1: Detect user emotion
    user_emotion = get_user_emotion(user_input)
    print(f"Detected Emotion: [{user_emotion}]")
    
    # Step 2: Filter movies by detected emotion
    filtered_movies = df_movies[
        df_movies['predicted_emotion'] == user_emotion
    ].copy()
    
    # Return message if no matching movies are found
    if filtered_movies.empty:
        return "No movies found for the detected emotion."
        
    print(f"Found {len(filtered_movies)} matching movies. Calculating similarity scores...")
    
    # Step 3: Convert user input and movie overviews into TF-IDF vectors
    user_vector = tfidf_vectorizer.transform([user_input])
    movie_vectors = tfidf_vectorizer.transform(
        filtered_movies['clean_overview']
    )
    
    # Step 4: Calculate cosine similarity
    similarity_scores = cosine_similarity(
        user_vector,
        movie_vectors
    ).flatten()
    
    # Step 5: Retrieve top-N highest similarity scores
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    
    # Step 6: Prepare recommendation results
    recommendations = filtered_movies.iloc[top_indices][
        ['Series_Title', 'Genre', 'Overview']
    ].copy()
    
    recommendations['similarity_score'] = similarity_scores[top_indices]
    
    return recommendations

In [8]:
def evaluate_precision_at_k(
    queries,
    top_k=5,
    threshold=0.05
):
    
    print(
        f"\n=== PRECISION@{top_k} "
        f"(Threshold={threshold}) ==="
    )
    
    total_precision = 0
    
    for i, query in enumerate(queries, 1):
        
        user_emotion = get_user_emotion(query)
        
        filtered_movies = df_movies[
            df_movies['predicted_emotion'] == user_emotion
        ].copy()
        
        if filtered_movies.empty:
            print(f"Q{i}: Precision = 0.00")
            continue
        
        user_vector = tfidf_vectorizer.transform(
            [query]
        )
        
        movie_vectors = tfidf_vectorizer.transform(
            filtered_movies['clean_overview']
        )
        
        similarity_scores = cosine_similarity(
            user_vector,
            movie_vectors
        ).flatten()
        
        top_indices = similarity_scores.argsort()[
            -top_k:
        ][::-1]
        
        top_scores = similarity_scores[top_indices]
        
        # Count relevant recommendations
        relevant_items = sum(
            score >= threshold
            for score in top_scores
        )
        
        precision = relevant_items / top_k
        
        total_precision += precision
        
        print(
            f"Q{i}: [{user_emotion}] "
            f"-> Relevant: {relevant_items}/{top_k} "
            f"| P@{top_k} = {precision:.2f}"
        )
    
    avg_precision = total_precision / len(queries)
    
    print("-" * 50)
    
    print(
        f"Average Precision@{top_k}: "
        f"{avg_precision:.2f}"
    )

print("Recommendation Engine Ready!")
print("-" * 50)

# =========================================================
# TEST RECOMMENDATION
# =========================================================
result = get_movie_recommendations(
    "I feel so lonely, isolated, and abandoned. "
    "I want to watch a drama movie about someone "
    "surviving alone in a difficult place.",
)

display(result)

# =========================================================
# PRECISION@K TEST QUERIES
# =========================================================
test_queries = [
    
    "I feel so lonely, isolated, and abandoned. "
    "I want to watch a drama movie about someone "
    "surviving alone in a difficult place.",
    
    "I am so angry right now. Someone betrayed me "
    "and I want a story about revenge and fighting back.",
    
    "I am super happy today! I just want a lighthearted "
    "comedy movie to celebrate with friends.",
    
    "I am terrified of the dark and monsters. "
    "Give me a horror movie with ghosts.",
    
    "I feel heartbroken after a breakup. "
    "I need a romantic story about moving on "
    "and finding new love."
]

evaluate_precision_at_k(
    test_queries,
    top_k=5,
    threshold=0.05
)

Recommendation Engine Ready!
--------------------------------------------------

User Input: 'I feel so lonely, isolated, and abandoned. I want to watch a drama movie about someone surviving alone in a difficult place.'
Detected Emotion: [Sadness]
Found 239 matching movies. Calculating similarity scores...


,Series_Title,Genre,Overview,similarity_score
228,Hachi: A Dog's Tale,"Biography, Drama, Family",A college professor bonds with an abandoned do...,0.129081
120,Singin' in the Rain,"Comedy, Musical, Romance",A silent film production company and cast make...,0.097616
747,Beasts of No Nation,"Drama, War","A drama based on the experiences of Agu, a chi...",0.095376
581,Under sandet,"Drama, History, War","In post-World War II Denmark, a group of young...",0.086496
992,The Jungle Book,"Animation, Adventure, Family",Bagheera the Panther and Baloo the Bear have a...,0.082158



=== PRECISION@5 (Threshold=0.05) ===
Q1: [Sadness] -> Relevant: 5/5 | P@5 = 1.00
Q2: [Anger] -> Relevant: 5/5 | P@5 = 1.00
Q3: [Joy] -> Relevant: 5/5 | P@5 = 1.00
Q4: [Fear] -> Relevant: 5/5 | P@5 = 1.00
Q5: [Sadness] -> Relevant: 5/5 | P@5 = 1.00
--------------------------------------------------
Average Precision@5: 1.00
